In [ ]:

import os
import sys
# Get the root directory (parent of 'notebooks/')
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root_dir)

In [ ]:
import torch
from model import train

In [3]:
DS = train.code_sim_datasets

In [ ]:
class conf: pass  # mock config

In [5]:
config = conf()
config.bs = 20
config.pretrained_bert_name = "huggingface/CodeBERTa-small-v1"
config.num_batches = 4
config.num_negatives = 2

In [113]:
from torch.utils.data import DataLoader


train_data, valid_data, test_data = train.code_sim_datasets.Create_CodeNet_triplet_dataset(
    tokenizer_name=config.pretrained_bert_name,
    tokenizer_max_length=256,
    num_negatives=config.num_negatives
)

train_loader = DataLoader(
    train_data, 
    batch_sampler=train.code_sim_datasets.CodeNetRandomTripletBatchSampler(
        pids=train_data.pids,
        num_batches=config.num_batches,
        num_pids_per_batch=config.bs
    ),
    collate_fn=train.code_sim_datasets.custom_collate_triplet
)

valid_loader = DataLoader(
    valid_data, batch_size=config.bs,
    sampler=train.code_sim_datasets.CodeNetDefaultTripletSampler(valid_data.pids),
    collate_fn=train.code_sim_datasets.custom_collate_triplet
)

test_loader = DataLoader(
    test_data, batch_size=config.bs,
    sampler=train.code_sim_datasets.CodeNetDefaultTripletSampler(test_data.pids),
    collate_fn=train.code_sim_datasets.custom_collate_triplet
)

Creating CodeNet dataset. Data type: triplet

        problem_id submission_id    status                    code
count      128400        128400    128400                  128400
unique        428        128400         2                  125189
top        p03711    s543499340  Accepted  print(int(input())**2)
freq          300             1     85600                      22 

Splitting dataset...
Train size: 102720, Valid size: 12840, Test size: 12840


In [161]:
assert not train_data.deterministic
assert valid_data.deterministic
assert test_data .deterministic
test_unique = lambda ds: len(ds.pids) == len(set(ds.pids))  
assert test_unique(train_data)
assert test_unique(valid_data)
assert test_unique(test_data )
assert valid_data["p02880"] == valid_data["p02880"]
assert test_data ["p02880"] == test_data ["p02880"]
a,p,ns = next(train_loader.__iter__())
assert a["input_ids"].shape[0] == config.bs
assert p["input_ids"].shape[0] == config.bs
assert ns["input_ids"].shape[0] == config.bs * config.num_negatives

In [153]:
print(ns["input_ids"].shape)

torch.Size([40, 256])


In [132]:
valid_data["p02880"] is valid_data["p02880"]

False

In [158]:
train_data["p02880"][0]["input_ids"]==train_data["p02880"][0]["input_ids"]

tensor([ True, False,  True,  True,  True,  True,  True,  True, False,  True,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False,  True, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True, 

In [159]:
a1,p1,ns1 = next(valid_loader.__iter__())
a2,p2,ns2 = next(valid_loader.__iter__())

In [160]:
assert torch.all(a1["input_ids"] == a2["input_ids"]).item()
assert torch.all(p1["input_ids"] == p2["input_ids"]).item()
assert torch.all(ns1["input_ids"] == ns2["input_ids"]).item()
assert torch.all(a1["attention_mask"] == a2["attention_mask"]).item()
assert torch.all(p1["attention_mask"] == p2["attention_mask"]).item()
assert torch.all(ns1["attention_mask"] == ns2["attention_mask"]).item()